In [ ]:
import os
import glob

import cupy as cp
import numpy as np
import pandas as pd
from itertools import product
import datetime
from tqdm import tqdm
import calendar

tqdm.pandas()

In [ ]:
# Google Colab으로 작업할 경우 실행
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials
from google.colab import drive


drive.mount('/content/drive')
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
my_drive = GoogleDrive(gauth)

In [ ]:
class TaxiTrip:
    """
    레스토랑 좌표와 Taxi Trip 좌표 사이의 거리를 계산하고, 1마일 이내 Trip 수와 Passgener 수를 조건에 따라 구분한 Column을 가지는 DataFrame을 만들어 최종적으로 CSV 파일로 내보내는 기능을 수행하는 Class입니다.
    해당 Class의 Method는 크게 원본 데이터를 불러와 필요한 변수들을 생성하는 초기화 함수 (__init__), 필요한 계산을 수행하는 함수 (getMonthlyData), 결과 DataFrame들을 합치고 다듬는 함수 (__concatDFList), 결과를 내보내는 함수 (exportDF) 등이 핵심이며, 나머지 Method들은 이 네 함수를 보조하는 역할을 합니다.
    
    코드와 동일한 폴더 내에 결과 파일을 저장하는 'output' 폴더와, 체크 포인트 파일을 저장하는 'checkpoint' 폴더, 그리고 Taxi Trip Data가 저장된 'tripdata' 폴더가 있어야 합니다.
    
    Instance를 생성하기 위해 필요한 Parameter들은 아래 __init__ 함수를 참조해주십시오.
    
    Methods:
        __init()__: 데이터를 불러와 필요한 형식으로 변환 및 저장합니다.
            ┗ __loadSavedCheckPoint(): 저장된 체크 포인트 파일이 있으면 불러옵니다.
            ┗ __initTripDF(): Taxi Trip 데이터를 불러와서 필요한 전처리를 수행합니다.
            ┗ __initTripMat(): Taxi Trip 데이터에서 계산에 필요한 칼럼만 선별하여 NumPy 또는 CuPy Array로 저장합니다.
            ┗ __initIdxBoundMat(): 각 관측 단위의 시간 범위에 들어가는 관측치들의 시작 Index와 끝 Index를 NumPy 또는 CuPy Array로 저장합니다. 
        getMonthlyData(lastColID:int=-1): 레스토랑마다 Vincenty Distance를 계산하고 결과를 DataFrame으로 저장합니다.
            ┗ __calculateVincentyDistance(resCord, miles:bool=True): 한 레스토랑의 좌표가 주어지면, 주어진 연-월의 모든 Taxi Trip의 Dropoff 또는 Pickup 좌표와의 거리를 Vincenty Formula로 계산합니다.
            ┗ __convertMatToDF(mat, id:int, yelpid: str, name:str, focal:int): 한 레스토랑에 대해 계산된 결과 Array를 pandas DataFrame 형식으로 변환합니다.
                ┗ @staticmethod __setPivotColName(colTuple:tuple): Pivot할 때 칼럼 이름을 지정합니다.
                ┗ __formatDateColumn(x: pd.Series): pd.Datetime 형식의 칼럼을 'YYYY-MM-DD' 형태의 string 형으로 변환합니다.
        __concatDFList(dfList:list[pd.DataFrame]): 결과 리스트에 있는 DataFrame들을 모두 하나의 DataFrame으로 합칩니다.
            ┗ __addAggCols(result:pd.DataFrame): 주어진 DataFrame에 비어있는 집계 칼럼들을 생성합니다.
            ┗ __addEmptyObs(result:pd.DataFrame): 결과를 저장한 DataFrame에 Taxi Trip이 없던 시간대 행을 추가하여 Balanced Panel 구조로 만듭니다.
        exportDF(checkPointTuple=(False, None)): 저장된 DataFrame을 CSV 형식으로 내보냅니다.
        @staticmethod getEmptyGDriveTrashBin(drive:GoogleDrive): Google Drive의 Trash Bin을 비웁니다.
    """
    
    ###############################
    # 초기화 관련 Method
    ###############################
    def __init__(self, year: int, month: int, colab:bool=False, checkPointFreq:int=1000, cupy:bool=True, daily=True, leave=False, part:int=-1):
        """
            주어진 연도-월의 Taxi Trip Distance를 계산하기 위해 필요한 초기화 작업을 수행합니다.
            'year'와 'month'는 필수 입력값이며, 나머지 Parameter들은 선택입니다. 기본값으로는 Colab을 이용하지 않고, 1,000개 레스토랑 결과마다 체크 포인트로 저장을 하며, 일 단위로 Dropoff를 계산합니다.
            체크 포인트 파일이 있다면 불러와 'outputDF' Attribute에 저장합니다.
            Taxi Trip 데이터를 불러와 'tripDF' Attribute에 저장 후 'initTripDF' Method로 초기화하고, 계산을 위한 형식으로 변환하여 'taxiCord', 'taxiTripDistanceMat', 'passengerCntMat' Attribute에 저장합니다. 마지막으로, 각 관측치의 시간 Index를 담은 Array를를 'idxBoundMat' Attribute에 저장합니다.
            레스토랑 좌표에서 Dropoff 좌표 또는 Pickup 좌표까지의 거리를 계산하는지에 따라 Taxi Trip 데이터에서 초점을 둘 Column이 달라지기 때문에, 이를 'targetCol' Attribute에 string 값으로 'dropoff' 또는 'pickup'을 저장합니다.
            'freqStr' Attribute에 string 값으로 일 단위의 관측 단위인 경우 'daily', 30분 단위의 관측 단위인 경우 'half-hour'를 저장합니다.
            'cursor' Attribute는 마지막으로 작업한 레스토랑 ID입니다. 어떠한 레스토랑에 대해서도 작업을 수행하지 않았을 경우 0의 값을 가집니다.
            
            Args:
                year (int): TaxiTrip Data의 연도
                month (int): TaxiTrip Data의 월
                colab (bool): Colab 이용 여부. 기본값: False
                checkPointFreq (int): 체크 포인트 도달 주기. 기본값: 1,000
                cupy (bool): CUDA 코어 이용 여부. 기본값: True
                daily (bool): 일 단위 계산 여부 (False 시 30분 단위 계산). 기본값: True
                leave (bool): Pickup 계산 여부 (False 시 Dropoff 계산). 기본값: False
                part (int): 30분 단위 계산 시 레스토랑 분할 번호. 기본값: -1
        """
        self.folderPath = '.' if not colab else '/content/drive/My Drive'
        self.resDF = pd.read_stata(f'{self.folderPath}/all_for_taxi_data_collection.dta')
        self.resDF['id'] = [i + 1 for i in range(len(self.resDF))]
        self.year = year
        self.month = month
        self.colab = colab
        self.cupy = cupy
        self.daily = daily
        self.part = part
        if not self.daily:
            if self.part == -1:
                print('[WARNING] part is not assigned. It''s set to 1.')
                self.part = 1
            partNumList = [len(self.resDF) // 4 + 1 - (i + 1) // 4 for i in range(4)]
            partIDList = []
            for i in range(4):
                if i < 3:
                    partIDList.append((1 + partNumList[i] * i, partNumList[i] * (i + 1)))
                else:
                    partIDList.append((partIDList[i - 1][1] + 1, partIDList[i - 1][1] + partNumList[i]))
            self.resDF = self.resDF[(self.resDF['id'] >= partIDList[self.part - 1][0]) & (self.resDF['id'] <= partIDList[self.part - 1][1])]
        self.lastResID = int(self.resDF['id'].max())
        self.leave = leave
        self.targetCol = 'dropoff' if not self.leave else 'pickup'
        self.freqStr = 'daily' if self.daily else 'half-hour'
        self.checkPointFreq = checkPointFreq
        self.outputDF, self.cursor = self.__loadSavedCheckPoint()
        print('Loading the taxi trip data set...')
        self.tripDF = pd.read_stata(f'{self.folderPath}/tripdata/tripdata_{year}-{month:02d}.dta')
        print('Initializing the dataframe...')
        self.__initTripDF()
        print('Initializing the arrays...')
        self.taxiCord, self.taxiTripDistanceMat, self.passengerCntMat = self.__initTripMat()
        self.idxBoundMat = self.__initIdxBoundMat()
        print('Done.')

    def __loadSavedCheckPoint(self):
        """
            체크 포인트 파일이 있는지 확인하고, 존재한다면 해당 파일을 불러와 Pandas DataFrame으로 저장합니다.
            또한, 체크 포인트 내 가장 큰 레스토랑의 ID도 저장합니다.
            마지막으로, 두 저장된 값을 반환합니다.
            
            Returns:
                tuple(pd.DataFrame, int)
        """
        fileName = f'{self.folderPath}/checkpoint/nearby_restaurant_taxi_{self.targetCol}s_{self.freqStr}_{self.year}-{self.month:02d}'
        if not self.daily:
            fileName += f'_part_{self.part}'
        filePath = f'{fileName}_cp_*.csv'
        matchingFiles = glob.glob(filePath)
        if len(matchingFiles) > 0: 
            cpDF = pd.read_csv(matchingFiles[0])
            lastCollectedID = cpDF['id'].max()
            print(f'Found the check point. Last ID: {lastCollectedID}')
            return cpDF, lastCollectedID
        else:
            cpDF = None
            lastCollectedID = 0
        return cpDF, lastCollectedID

    def __initTripDF(self):
        """
            저장된 초기 tripDF Attribute에 대해 전처리를 수행합니다.
            조건에 맞지 않은 부적절한 관측치들을 삭제하고, 행을 Timestamp 순으로 정렬합니다.
        """
        self.tripDF['pickup_dt'] = pd.to_datetime(self.tripDF['pickup_dt'], format='%Y-%m-%d %H:%M:%S')
        self.tripDF['dropoff_dt'] = pd.to_datetime(self.tripDF['dropoff_dt'], format='%Y-%m-%d %H:%M:%S')

        invalidCond1 = self.tripDF['dropoff_dt'] - self.tripDF['pickup_dt'] >= datetime.timedelta(hours=3) # 도착 시간과 출발 시간이 3시간 이상 차이나는 경우
        invalidCond2 = self.tripDF['trip_distance'] >= 100 # 주행 거리가 100마일 이상인 경우
        invalidCond3 = self.tripDF['passenger_count'] == 0 # 탑승자수가 0명인 경우
        self.tripDF = self.tripDF.loc[(~invalidCond1) & (~invalidCond2) & (~invalidCond3)].reset_index(drop=True)

        # 주어진 월의 마지막 일 이후 Drop-off는 기준 월의 마지막 일 오후 11시 59분 59초로 설정.
        lastDay = calendar.monthrange(self.year, self.month)[1]
        lastTimeStamp = pd.Timestamp(f'{self.year}-{self.month}-{lastDay} 23:59:59')
        self.tripDF['dropoff_dt'] = self.tripDF['dropoff_dt'].apply(lambda x: min(x, lastTimeStamp))

        self.tripDF[f'{self.targetCol}_date'] = self.tripDF[f'{self.targetCol}_dt'].dt.date
        if not self.daily:
            self.tripDF['hour'] = self.tripDF[f'{self.targetCol}_dt'].dt.hour
            self.tripDF['minute'] = self.tripDF[f'{self.targetCol}_dt'].dt.minute
            self.tripDF['minute_g'] = 0
            self.tripDF.loc[self.tripDF['minute'] >= 30, 'minute_g'] = 30
        self.tripDF = self.tripDF.sort_values(by=[f'{self.targetCol}_dt', f'{self.targetCol}_lat', f'{self.targetCol}_lon']).reset_index(drop=True)

    def __initTripMat(self):
        """
            tripDF로부터 거리 및 Passenger 수 계산에 필요한 값만 추출하여 NumPy 또는 CuPy Array로 저장합니다.
            구체적으로, Dropoff 또는 Pickup의 좌표를 담고 있는 Array, Trip Distance를 담고 있는 Array, Passenger Count를 담고 있는 Array
            등 세 가지 Array의 Tuple을 반환합니다.
            
            Return:
                tuple(np.array | cp.array, np.array | cp.array, np.array | cp.array)
        """
        if self.cupy:
            taxiCord = cp.array(self.tripDF[[f'{self.targetCol}_lat', f'{self.targetCol}_lon']].to_numpy(dtype=float), dtype=cp.float32)
            taxiTripDistanceMat = cp.array(self.tripDF['trip_distance'].to_numpy(dtype=float), dtype=cp.float32)
            passengerCntMat = cp.array(self.tripDF['passenger_count'].to_numpy(dtype=int))
        else:
            taxiCord = self.tripDF[[f'{self.targetCol}_lat', f'{self.targetCol}_lon']].to_numpy(dtype=float)
            taxiTripDistanceMat = self.tripDF['trip_distance'].to_numpy(dtype=float)
            passengerCntMat =  self.tripDF['passenger_count'].to_numpy(dtype=int)
        return taxiCord, taxiTripDistanceMat, passengerCntMat

    def __initIdxBoundMat(self):
        """
            시간순으로 정렬된 tripDF의 각 Index에 대해, 관측 단위 (일 또는 30분 단위) 범위에 대응하는 시간 Index를 담은 Array를 반환합니다.
            시간 Index는 Daily 단위로 계산한다면 Day 값을 가지며, 30분 단위로 계산한다면 다음과 같은 Fractional Day 값을 가집니다:
                Day + (Hour + (Minute / 60)) / 24
            예를 들어, tripDF.loc[4568, 'dropoff_dt'] 값이 '2016-01-01 11:23:21'이라면, 
            Daily 단위로 계산할 때 idxBountMat[4568]의 값은 23이며, 30분 단위로 계산할 때는 23.4743055... 값을 가집니다.

            Return:
                np.array | cp.array
        """
        startDateStr = f'{self.year}-{self.month}-01'
        lastDay = (pd.Timestamp(f'{self.year}-{self.month}-01') + pd.offsets.MonthEnd(1)).day
        endDateStr = f'{self.year}-{self.month}-{lastDay}'
        if self.daily:
            maxIndexSeries = self.tripDF.groupby(f'{self.targetCol}_date').apply(lambda x: x.index.max(), include_groups=False)
        else:
            maxIndexSeries = self.tripDF.groupby([f'{self.targetCol}_date', 'hour', 'minute_g'], group_keys=False).apply(lambda x: x.index.max(), include_groups=False)
        idxList = list(maxIndexSeries.index)
        idxBoundDict = {}
        if self.daily:
            dateList = pd.date_range(start=startDateStr, end=endDateStr).date.tolist()
        else:
            endDateStr += ' 23:59:59'
            dateList = pd.date_range(start=startDateStr, end=endDateStr, freq='30min').to_list()
        for date in dateList:
            if self.daily:
                endIdx = maxIndexSeries[date]
                if date == maxIndexSeries.index.min():
                    startIdx = 0
                else:
                    currentIdxPos = idxList.index(date)
                    pvIdx = idxList[currentIdxPos - 1]
                    startIdx = maxIndexSeries[pvIdx] + 1
                idxBoundDict[date] = (startIdx, endIdx)
            else:
                if (date.date(), date.hour, date.minute) in maxIndexSeries:
                    endIdx = maxIndexSeries[date.date(), date.hour, date.minute]
                    if (date.date(), date.hour, date.minute) == maxIndexSeries.index.min():
                        startIdx = 0
                    else:
                        currentIdxPos = idxList.index((date.date(), date.hour, date.minute))
                        pvIdx = idxList[currentIdxPos - 1]
                        startIdx = maxIndexSeries[pvIdx] + 1
                    idxBoundDict[date] = (startIdx, endIdx)

        if self.cupy:
            mat = cp.array(list(self.tripDF.index))
            bounds = cp.array(list(idxBoundDict.values()))
            lb = cp.array([b[0] for b in idxBoundDict.values()])
            ub = cp.array([b[1] for b in idxBoundDict.values()])

            idxStart = cp.searchsorted(lb, mat, side='right') - 1
            validMask = (idxStart >= 0) & (idxStart < len(bounds)) & (mat >= lb[idxStart]) & (mat <= ub[idxStart])
            choiceListArray = cp.array([key.day if self.daily else key.day + (key.hour + (key.minute / 60)) / 24 for key in idxBoundDict.keys()])

            idxBoundMat = cp.zeros_like(mat, dtype=cp.float32)
            idxBoundMat[validMask] = choiceListArray[idxStart[validMask]]

            cp.get_default_memory_pool().free_all_blocks()
            cp.get_default_pinned_memory_pool().free_all_blocks()
        else:
            mat = np.array(list(self.tripDF.index))
            bounds = np.array(list(idxBoundDict.values()))
            lb = np.array([b[0] for b in idxBoundDict.values()])
            ub = np.array([b[1] for b in idxBoundDict.values()])

            idxStart = np.searchsorted(lb, mat, side='right') - 1
            validMask = (idxStart >= 0) & (idxStart < len(bounds)) & (mat >= lb[idxStart]) & (mat <= ub[idxStart])
            choiceListArray = np.array([key.day if self.daily else key.day + (key.hour + (key.minute / 60)) / 24 for key in idxBoundDict.keys()])

            idxBoundMat = np.zeros_like(mat)
            idxBoundMat[validMask] = choiceListArray[idxStart[validMask]]

        return idxBoundMat

    ###############################
    # 계산 관련 Method
    ###############################
    def getMonthlyData(self, lastColID=-1):
        """
            각 레스토랑에 대해 아래의 작업을 수행합니다.
                1. calculateVincentyDistance Method를 호출하여 레스토랑 좌표와 Taxi Trip내 모든 좌표 사이의 거리를 계산하고, 거리가 1마일 이하인 Index만 고려하여 'distanceMat' Array 변수에 저장합니다.
                2. 레스토랑과의 거리가 Circle 또는 Band에 들어오는 각각의 경우에 대해 1부터 6까지의 값을 가지는 Mask Array 'resDistanceMaskMat'을 생성합니다. 만약 어느 조건에도 만족하지 않는 경우 0의 값을 가집니다.
                3. Taxi Trip Distance가 특정 조건을 만족하는 각각의 경우에 대해 1부터 5까지의 값을 가지는 Mask Array 'taxiTripDistanceMaskMat'을 생성합니다. 마찬가지로 어느 조건에도 만족하지 않는 경우 0의 값을 가집니다.
                4. idxBountMat Attribute에서 거리가 1마일 이하인 Index만 고려하여 'thisIdxBountMat'에 저장합니다. 
                5. (thisIdxBoundMat, taxiTripDistanceMaskMat, resDistanceMaskMat) 각각의 Tuple 값을 하나의 숫자에 일대일 대응시킨 Hash Key 값을 만들고, 'hashKeys' Array에 저장합니다.
                6. 각각의 Hash Key마다 Passenger 수와 Trip 수를 계산하여 'sumPassengerCntMat'과 'counts' Array에 저장합니다.
                7. (thisIdxBoundMat, taxiTripDistanceMaskMat, resDistanceMaskMat, counts, sumPassengerCntMat)을 Column Stack한 'finalMat' Array를 생성합니다.
                8. 'convertMatToDF' Method를 호출하여, 'finalMat' Array를 Pandas DataFrame으로 변환하고, DataFrame List인 'dfList'에 Append합니다.
            체크 포인트에 도달하거나, 모든 레스토랑에 대해 작업이 완료된 경우, 'concatDFList' Method를 호출하여 'dfList'에 있는 DataFrame들을 Concat하고 'outputDF' Attribute에 저장합니다.

            Args:
                lastColID (int) = -1: 마지막으로 수집할 레스토랑 ID입니다. 기본값은 -1로, 처음부터 수집합니다.
        """
        dfList = []
        if lastColID == -1:
            lastColID = self.lastResID
        if type(lastColID) != int or lastColID < 1 or lastColID > self.lastResID:
            print('Invalid Parameter.')
            return None
        if self.outputDF is not None and lastColID in self.outputDF['id'].unique():
            print('Nothing to do.')
            return None
        if not self.daily:
          descStr = f'Year: {self.year}, Month: {self.month}, Part: {self.part}'
        else:
          descStr = f'Year: {self.year}, Month: {self.month}'
        for thisResTuple in tqdm(self.resDF.itertuples(index=False), total=len(self.resDF), miniters=0, desc=descStr):
            thisID = thisResTuple.id
            if thisID < self.cursor + 1:
                continue
            if thisID > lastColID:
                print('Reached the last ID.')
                break
            if self.cupy:
                resCord = cp.array([[thisResTuple.lat_final, thisResTuple.lon_final]])
            else:
                resCord = np.array([[thisResTuple.lat_final, thisResTuple.lon_final]])
            distanceMat = self.__calculateVincentyDistance(resCord, True) * 5280
            distMask = distanceMat <= 5280
            distanceMat = distanceMat[distMask]

            if self.cupy:
                resDistanceGroupList = cp.array([(distanceMat <= 100),
                    ((100 < distanceMat) & (distanceMat <= 200)),
                    ((200 < distanceMat) & (distanceMat <= 300)),
                    ((300 < distanceMat) & (distanceMat <= 400)),
                    ((400 < distanceMat) & (distanceMat <= 500)),
                    ((500 < distanceMat) & (distanceMat <= 600))])
                resDistanceMaskMat = cp.select(resDistanceGroupList, cp.array([1, 2, 3, 4, 5, 6]), default=0)
            else:
                resDistanceGroupList = np.array([(distanceMat <= 100),
                    ((100 < distanceMat) & (distanceMat <= 200)),
                    ((200 < distanceMat) & (distanceMat <= 300)),
                    ((300 < distanceMat) & (distanceMat <= 400)),
                    ((400 < distanceMat) & (distanceMat <= 500)),
                    ((500 < distanceMat) & (distanceMat <= 600))])
                resDistanceMaskMat = np.select(resDistanceGroupList, np.array([1, 2, 3, 4, 5, 6]), default=0)

            thisTaxiTripDistanceMat = self.taxiTripDistanceMat[distMask]
            if self.cupy:
                taxiTripGroupList = cp.array([thisTaxiTripDistanceMat <= 1,
                                        ((1 < thisTaxiTripDistanceMat) & (thisTaxiTripDistanceMat <= 2)),
                                        ((2 < thisTaxiTripDistanceMat) & (thisTaxiTripDistanceMat <= 3)),
                                        ((3 < thisTaxiTripDistanceMat) & (thisTaxiTripDistanceMat <= 4)),
                                        (4 < thisTaxiTripDistanceMat)])
                taxiTripDistanceMaskMat = cp.select(taxiTripGroupList, cp.array([1, 2, 3, 4, 5]), default=0)
            else:
                taxiTripGroupList = np.array([thisTaxiTripDistanceMat <= 1,
                                        ((1 < thisTaxiTripDistanceMat) & (thisTaxiTripDistanceMat <= 2)),
                                        ((2 < thisTaxiTripDistanceMat) & (thisTaxiTripDistanceMat <= 3)),
                                        ((3 < thisTaxiTripDistanceMat) & (thisTaxiTripDistanceMat <= 4)),
                                        (4 < thisTaxiTripDistanceMat)])
                taxiTripDistanceMaskMat = np.select(taxiTripGroupList, np.array([1, 2, 3, 4, 5]), default=0)

            thisIdxBoundMat = self.idxBoundMat[distMask]
            hashKeys = thisIdxBoundMat * 10000 + taxiTripDistanceMaskMat * 10 + resDistanceMaskMat
            if self.cupy:
                sortIdx = cp.argsort(hashKeys)
            else:
                sortIdx = np.argsort(hashKeys)
            sortedKeys = hashKeys[sortIdx]
            thisPassengerCntMat = self.passengerCntMat[distMask]
            sortedPassengerCntMat = thisPassengerCntMat[sortIdx]

            if self.cupy:
                uniqueKey, firstIdx, counts = cp.unique(sortedKeys, return_index=True, return_counts=True)
                cumsumPassengerCnt = cp.cumsum(sortedPassengerCntMat)
                sumPassengerCntMat = cp.diff(cp.concatenate((cp.array([0]), cumsumPassengerCnt[firstIdx + counts - 1])))
            else:
                uniqueKey, firstIdx, counts = np.unique(sortedKeys, return_index=True, return_counts=True)
                cumsumPassengerCnt = np.cumsum(sortedPassengerCntMat)
                sumPassengerCntMat = np.diff(np.concatenate((np.array([0]), cumsumPassengerCnt[firstIdx + counts - 1])))
            oriIdx = sortIdx[firstIdx]

            if self.cupy:
                finalMat = cp.column_stack((resDistanceMaskMat[oriIdx], taxiTripDistanceMaskMat[oriIdx], thisIdxBoundMat[oriIdx], counts, sumPassengerCntMat))
            else:
                finalMat = np.column_stack((resDistanceMaskMat[oriIdx], taxiTripDistanceMaskMat[oriIdx], thisIdxBoundMat[oriIdx], counts, sumPassengerCntMat))
            thisDF = self.__convertMatToDF(finalMat, thisID, thisResTuple.yelpid, thisResTuple.name, thisResTuple.focal)

            dfList.append(thisDF)
            self.cursor += 1

            if thisID % self.checkPointFreq == 0:
                checkPointNum = thisID // self.checkPointFreq
                #print(f'Reached check point #{checkPointNum}... Saving it..')
                self.__concatDFList(dfList)
                self.exportDF((True, checkPointNum))
                dfList = []
            if self.cupy:
                del resCord, distMask, distanceMat, resDistanceGroupList, resDistanceMaskMat, thisTaxiTripDistanceMat, taxiTripGroupList, taxiTripDistanceMaskMat, thisIdxBoundMat, hashKeys, sortIdx, sortedKeys,  sortedPassengerCntMat, uniqueKey, firstIdx, counts, cumsumPassengerCnt, sumPassengerCntMat, oriIdx, finalMat
                cp.get_default_memory_pool().free_all_blocks()
                cp.get_default_pinned_memory_pool().free_all_blocks()

        if len(dfList) > 0:
            self.__concatDFList(dfList)
            
    def __calculateVincentyDistance(self, resCord, miles:bool=False):
        """
            주어진 레스토랑 좌표와 Taxi Trip의 모든 좌표와의 거리를 계산한 Array를 반환합니다.
            알고리즘 특성상 완전한 Vectorization은 어려워 부분적으로만 적용되었습니다.
            
            Args:
                resCord (np.array | cp.array): 계산하고자하는 레스토랑 좌표를 담은 Array 입니다.
                miles (bool) = False: 거리를 mile로 반환할지 여부입니다. True이면 mile로 반환하고, False이면 km로 반환합니다.
                
            Returns:
                np.array | cp.array
        """
        _a = 6378137.0
        _f = 1 / 298.257223563
        _b = (1 - _f) * _a
        if self.cupy:
            mask1 = cp.all(abs(self.taxiCord - resCord) < 1e-12, axis=1)

            U1 = cp.arctan((1 - _f) * cp.tan(cp.radians(resCord[:, 0])))
            U2 = cp.arctan((1 - _f) * cp.tan(cp.radians(self.taxiCord[:, 0])))

            L = cp.radians(self.taxiCord[:, 1] - resCord[:, 1])

            sinU1, cosU1 = cp.sin(U1), cp.cos(U1)
            sinU2, cosU2 = cp.sin(U2), cp.cos(U2)
        else:
            mask1 = np.all(abs(self.taxiCord - resCord) < 1e-12, axis=1)

            U1 = np.arctan((1 - _f) * np.tan(np.radians(resCord[:, 0])))
            U2 = np.arctan((1 - _f) * np.tan(np.radians(self.taxiCord[:, 0])))

            L = np.radians(self.taxiCord[:, 1] - resCord[:, 1])

            sinU1, cosU1 = np.sin(U1), np.cos(U1)
            sinU2, cosU2 = np.sin(U2), np.cos(U2)

        lamb = L

        iterLimit = 100
        for _ in range(iterLimit):
            if self.cupy:
                sinLamb, cosLamb = cp.sin(lamb), cp.cos(lamb)
                sinSigma = cp.hypot(cosU2 * sinLamb, + cosU1 * sinU2 - sinU1 * cosU2 * cosLamb)
            else:
                sinLamb, cosLamb = np.sin(lamb), np.cos(lamb)
                sinSigma = np.hypot(cosU2 * sinLamb, + cosU1 * sinU2 - sinU1 * cosU2 * cosLamb)

            mask2 = sinSigma == 0

            cosSigma = sinU1 * sinU2 + cosU1 * cosU2 * cosLamb
            if self.cupy:
                sigma = cp.arctan2(sinSigma, cosSigma)
            else:
                sigma = np.arctan2(sinSigma, cosSigma)

            sinAlpha = (cosU1 * cosU2 * sinLamb) / sinSigma
            cos2Alpha = 1 - sinAlpha ** 2

            mask3 = cos2Alpha == 0
            cos2SigmaM = cosSigma - 2 * sinU1 * sinU2 / cos2Alpha
            cos2SigmaM[mask3] = 0

            C = (_f / 16) * cos2Alpha * (4 + _f * (4 - 3 * cos2Alpha))
            if self.cupy:
                lambNew = L + (1 - C) * _f * sinAlpha * (sigma + C * sinSigma * (cos2SigmaM + C * cosSigma * (-1 + 2 * cp.cos(2 * sigma)**2)))
            else:
                lambNew = L + (1 - C) * _f * sinAlpha * (sigma + C * sinSigma * (cos2SigmaM + C * cosSigma * (-1 + 2 * np.cos(2 * sigma)**2)))

            mask3 = mask1 | mask2 | (abs(lambNew - lamb) < 1e-12)
            if self.cupy:
                if cp.all(mask3):
                    break
            else:
                if np.all(mask3):
                    break

            lamb = lambNew

        U2 = cos2Alpha * ((_a**2 - _b**2) / (_b**2))
        A = 1 + (U2 / 16384) * (4096 + U2 * (-768 + U2 * (320 - 175 * U2)))
        B = (U2 / 1024) * (256 + U2 * (-128 + U2 * (74 - 47 * U2)))
        deltaSigma = B * sinSigma  * (cos2SigmaM + (B / 4) * (cosSigma * (-1 + 2 * cos2SigmaM**2) - (B / 6) * cos2SigmaM * (-3 + 4 * sinSigma**2) * (-3 + 4 * cos2SigmaM ** 2)))

        if self.cupy:
            distance = cp.zeros_like(L)
        else:
            distance = np.zeros_like(L)
        distance = (_b * A * (sigma - deltaSigma)) / 1000
        distance[mask1 | mask2] = 0
        if miles:
            distance[~mask1 & ~mask2] *= 0.621371192237334
        return distance

    def __convertMatToDF(self, mat, id:int, yelpid: str, name:str, focal:int):
        """
            Array로 저장된 결과를 Pandas DataFrame으로 변환합니다.
            해당 Method로 반환되는 DataFrame에는 조건에 만족하는 Trip이 없는 Column을 포함하지 않습니다.
            예를 들어, 레스토랑과 Dropoff 또는 Pickup 거리가 200피트 초과, 300 피트 이하인 경우가 없는 경우, trip_c3, trip_c3_d1, ... trip_c3_d5 Column들은 존재하지 않습니다.
            
            Args:
                mat (np.array | cp.array): 결과를 저장한 Array입니다.
                id (int): 레스토랑의 ID입니다.
                yelpid (str): 레스토랑의 Yelp ID입니다.
                name (str): 레스토랑의 이름입니다.
                focal (int): 레스토랑의 Focal 여부입니다.

            Returns:
                pd.DataFrame
        """
        if self.cupy:
            df = pd.DataFrame(mat.get(), columns=['res_distance', 'trip_distance', 'date', 'trip', 'pass'])
        else:
            df = pd.DataFrame(mat, columns=['res_distance', 'trip_distance', 'date', 'trip', 'pass'])
        df['res_distance'] = df['res_distance'].astype(int)
        df['trip_distance'] = df['trip_distance'].astype(int)
        df['trip'] = df['trip'].astype(int)
        df['pass'] = df['pass'].astype(int)
        if not self.daily:
            df['hm'] = round((df['date'] % 1) * 24, 1) / 1
            df['hour'] = (df['hm'] / 1).astype(int)
            df['minute'] = (round((df['hm'] % 1) * 60)).astype(int)
            df['date'] = (df['date'] / 1).astype(int)
            dfPivot = df.pivot(index=['date', 'hour', 'minute'], columns=['res_distance', 'trip_distance'], values=['trip', 'pass'])
        else:
            df['date'] = df['date'].astype(int)
            dfPivot = df.pivot(index=['date'], columns=['res_distance', 'trip_distance'], values=['trip', 'pass'])
        dfPivot.columns = [self.__setPivotColName(col) for col in dfPivot.columns]
        dfPivot = dfPivot.drop(columns=[colName for colName in dfPivot.columns if colName.startswith('-1')]).reset_index()
        dfPivot = dfPivot[dfPivot['date'] > 0].reset_index(drop=True)
        dfPivot['id'] = id
        dfPivot['yelpid'] = yelpid
        dfPivot['name'] = name
        dfPivot['has_trip'] = 1
        dfPivot['focal'] = focal
        colNameList = list(dfPivot.columns)
        fullColList = [f'{measure}_c{i}_d{j}' for i, j, measure in product(range(0, 6 + 1), range(1, 5 + 1), ['trip', 'pass'])]
        for colName in fullColList:
            if not (colName in colNameList):
                dfPivot[colName] = 0
        infoColList = ['id', 'yelpid', 'name', 'focal', 'has_trip', 'date']
        if not self.daily:
            infoColList.extend(['hour', 'minute'])
        dfPivot = dfPivot[infoColList + fullColList]
        dfPivot.fillna(0, inplace=True)
        dfPivot[fullColList] = dfPivot[fullColList].astype(int)
        dfPivot['date'] = dfPivot['date'].apply(self.__formatDateColumn)
        return dfPivot

    @staticmethod
    def __setPivotColName(colTuple: tuple):
        """
            'finalMat' Array를 DataFrame으로 변환 후, 원하는 형태의 DataFrame을 만들어주기 위해서 Pivot을 수행합니다.
            이 과정에서 만들어지는 Column들의 이름을 지정합니다.
            
            Args:
                colTuple (tuple): (measure, resDistanceMask, tripDistanceMask) 값을 가지는 Tuple입니다. 

            Returns:
                str
        """
        measure, resDistanceMask, tripDistanceMask = colTuple
        if int(resDistanceMask) >= 0:
            newColName = f'{measure}_c{resDistanceMask}_d{tripDistanceMask}'
        else:
            newColName = measure
        return newColName

    def __formatDateColumn(self, x):
        """
            Day 값을 'YYYY-MM-DD' 형태로 변환합니다.
            
            Args:
                x (int): Day 값입니다.

            Returns:
                str
        """
        return f'{self.year}-{self.month:02d}-{x:02d}'

    ###############################
    # DataFrame 후처리 관련 Method
    ###############################
    def __concatDFList(self, dfList: list[pd.DataFrame]):
        """
            DataFrame List에 있는 DataFrame들을 하나로 합칩니다.
            그 과정에서 'addAggCols' Method를 호출하여 집계 Column을 추가하고, 'addEmptyObs' Methods를 호출하여 해당 레스토랑 근처 Taxi Trip이 없던 시간대에 대한 행을 추가하여, Balanced Panel 구조를 가지도록 합니다.
            합쳐진 DataFrame은 'outputDF' Attribute에 저장합니다.
            
            Args:
               dfList (list[pd.DataFrame]): DataFrame List입니다.
        """
        thisOutputDF = pd.concat(dfList)
        thisOutputDF = self.__addAggCols(thisOutputDF)
        thisOutputDF = self.__addEmptyObs(thisOutputDF)
        if self.outputDF is not None:
            self.outputDF = pd.concat([self.outputDF, thisOutputDF])
        else:
            self.outputDF = thisOutputDF

    def __addAggCols(self, result:pd.DataFrame):
        """
            결과를 저장한 DataFrame에 비어있는 집계 Column을 추가합니다.
            
            Args:
              result (pd.DataFrame): 비어있는 집계 Column을 추가할 DataFrame입니다.
              
            Returns:
                pd.DataFrame
        """
        cols = result.columns
        for agg in range(-1, 6 + 1):
            if agg == -1:
                if 'trip' not in result.columns:
                    result['trip'] = 0
                    tripCols = [col for col in cols if col.find('trip') != -1 and col != 'trip' and col != 'has_trip']
                    for tripCol in tripCols:
                        result['trip'] += result[tripCol]
                if 'pass' not in result.columns:
                    result['pass'] = 0
                    passCols = [col for col in cols if col.find('pass') != -1]
                    for passCol in passCols:
                        result['pass'] += result[passCol]
            else:
                if f'trip_c{agg}' not in result.columns:
                    result[f'trip_c{agg}'] = 0
                    tripCols = [col for col in cols if col.find(f'trip_c{agg}') != -1]
                    for tripCol in tripCols:
                        result[f'trip_c{agg}'] += result[tripCol]
                if f'pass_c{agg}' not in result.columns:
                    result[f'pass_c{agg}'] = 0
                    tripCols = [col for col in cols if col.find(f'pass_c{agg}') != -1]
                    for tripCol in tripCols:
                        result[f'pass_c{agg}'] += result[tripCol]

        timeCols = ['date']
        if not self.daily:
            timeCols.extend(['hour', 'minute'])
        resultCols = ['id', 'yelpid', 'name', 'focal', 'has_trip'] + timeCols + ['trip', 'pass']
        for i in range(0, 6 + 1):
            resultCols.extend([f'trip_c{i}', f'pass_c{i}'])
            for j in range(1, 5 + 1):
                resultCols.extend([f'trip_c{i}_d{j}', f'pass_c{i}_d{j}'])

        result = result[resultCols]
        return result

    def __addEmptyObs(self, result: pd.DataFrame):
        """
            결과를 저장한 DataFrame에 Taxi Trip이 없던 시간대 행을 추가하여 Balanced Panel 구조로 만듭니다.
            
            Args:
              result (pd.DataFrame): 대상이 되는 DataFrame입니다.
              
            Returns:
                pd.DataFrame
        """
        result['date'] = pd.to_datetime(result['date'])
        monthStart = result['date'].min().replace(day=1)
        monthEnd = result['date'].max().replace(day=1) + pd.DateOffset(months=1) - pd.DateOffset(days=1)
        yelpIDList = result['yelpid'].unique()
        if self.daily:
            allDates = pd.date_range(start=monthStart, end=monthEnd, freq='D')
            fullIndex = pd.MultiIndex.from_product([yelpIDList, allDates], names=['yelpid', 'date'])
            result = result.set_index(['yelpid', 'date'])
        else:
            dateList = pd.date_range(start=monthStart, end=monthEnd)
            hourList = [i for i in range(0, 23 + 1)]
            minList = [0, 30]
            dateTupleList = list(product(yelpIDList, dateList, hourList, minList))
            fullIndex = pd.MultiIndex.from_tuples(dateTupleList, names=['yelpid', 'date', 'hour', 'minute'])
            result = result.set_index(['yelpid', 'date', 'hour', 'minute'])
        result = result.reindex(fullIndex).reset_index()
        id_map = self.resDF.set_index('yelpid')['id'].to_dict()
        name_map = self.resDF.set_index('yelpid')['name'].to_dict()
        focal_map = self.resDF.set_index('yelpid')['focal'].to_dict()
        result['id'] = result['id'].fillna(result['yelpid'].map(id_map))
        result['name'] = result['name'].fillna(result['yelpid'].map(name_map))
        result['focal'] = result['focal'].fillna(result['yelpid'].map(focal_map))
        result.fillna(0, inplace=True)
        timeCols = ['date']
        if not self.daily:
            timeCols.extend(['hour', 'minute'])
        infoCols = ['id', 'yelpid', 'name', 'focal', 'has_trip'] + timeCols + ['trip', 'pass']
        dataCols = [col for col in result.columns if col not in infoCols]
        result = result[infoCols + dataCols]
        result['date'] = pd.to_datetime(result['date']).dt.date
        return result

    ###############################
    # CSV 내보내기 관련 Method
    ###############################
    def exportDF(self, checkPointTuple=(False, None)):
        """
            'outputDF' Attribute의 DataFrame을 CSV 파일로 내보냅니다. 체크 포인트에 도달하여 내보낼 경우, 이전 체크 포인트 결과 파일을 삭제하고, Google Drive Trash Bin을 비웁니다. 모든 레스토랑에 대한 계산이 끝나 최종 결과를 내보낼 경우, 체크 포인트 결과 파일을 삭제합니다.
            
            Args:
              checkPointTuple (tuple) = (False, None): 체크 포인트에 대한 정보를 담은 Tuple입니다. 첫번째 Element는 체크 포인트에 도달하여 내보내는지 여부이며, 두 번째 Element는 체크 포인트에 도달하여 내보낼 경우 몇 번째 체크 포인트인지 나타내는 값입니다.
        """
        delCP = False
        if self.outputDF['id'].max() == self.lastResID:
            delCP = True
        fileName = f'nearby_restaurant_taxi_{self.targetCol}s_{self.freqStr}_{self.year}-{self.month:02d}'
        if not self.daily:
            fileName += f'_part_{self.part}'
        detailedPath = 'output'
        checkPoint = checkPointTuple[0]
        exportFileName = fileName
        if checkPoint:
            filePath = f'{self.folderPath}/checkpoint/{fileName}_cp_*.csv'
            matchingFiles = glob.glob(filePath)
            if len(matchingFiles) > 0:
                for file in matchingFiles:
                    os.remove(file)
                    if self.colab:
                        self.getEmptyGDriveTrashBin(my_drive)
            checkPointNum = checkPointTuple[1]
            exportFileName += f'_cp_{checkPointNum}'
            detailedPath = 'checkpoint'

        intCols = ['id', 'focal', 'has_trip', 'trip', 'pass']
        for i in range(0, 6 + 1):
            intCols.extend([f'trip_c{i}', f'pass_c{i}'])
            for j in range(1, 5 + 1):
                intCols.extend([f'trip_c{i}_d{j}', f'pass_c{i}_d{j}'])
        for col in intCols:
          self.outputDF[col] = self.outputDF[col].astype(int)

        self.outputDF.to_csv(f'{self.folderPath}/{detailedPath}/{exportFileName}.csv', index=False)
        if delCP:
            filePath = f'{self.folderPath}/checkpoint/{fileName}_cp_*.csv'
            matchingFiles = glob.glob(filePath)
            if len(matchingFiles) > 0:
                for file in matchingFiles:
                    os.remove(file)
                    if self.colab:
                        self.getEmptyGDriveTrashBin(my_drive)

    """
    ## Colab에서 작업하지 할 경우 Comments 삭제
    @staticmethod
    def getEmptyGDriveTrashBin(drive:GoogleDrive):
      for file in drive.ListFile({'q': "trashed = true"}).GetList():
        try:
          file.Delete()
        except:
          pass
    """

### Colab에서 작업 시 예시

In [ ]:
def getYearMonthPair(sessionID, col='dropoff', freq='daily'):
    """
        아직 수집이 완료되지 않은 연-월을 조회하고, List로 반환합니다.
        Colab에서 작업할 경우에만 사용합니다.

        Args:
            sessionID (int): Google Colab의 각 Session을 구분하기 위한 ID입니다.
            col (str): 계산할 대상이 되는 Column 이름입니다. 'dropoff' 또는 'pickup'입니다.
            freq (str): 관측 단위입니다. 일 단위의 경우 'daily', 30분 단위인 경우 'half-hour'입니다.
            
        Return:
            list
    """
    pairList = []
    if 1 <= sessionID <= 3:
        if sessionID == 1:
            for year, month in product(range(2009, 2011 + 1), range(1, 12 + 1)):
                if year == 2011 and month > 6:
                    continue
                if os.path.exists(f'/content/drive/My Drive/output/nearby_restaurant_taxi_{col}s_{freq}_{year}-{month:02d}.csv'):
                  continue
                pairList.append((year, month))
        elif sessionID == 2:
            for year, month in product(range(2011, 2013 + 1), range(1, 12 + 1)):
                if year == 2011 and month < 7:
                    continue
                if os.path.exists(f'/content/drive/My Drive/output/nearby_restaurant_taxi_{col}s_{freq}_{year}-{month:02d}.csv'):
                  continue
                pairList.append((year, month))
        elif sessionID == 3:
            for year, month in product(range(2014, 2016 + 1), range(1, 12 + 1)):
                if year == 2016 and month > 6:
                    continue
                if os.path.exists(f'/content/drive/My Drive/output/nearby_restaurant_taxi_{col}s_{freq}_{year}-{month:02d}.csv'):
                  continue
                pairList.append((year, month))
        print(f'This session take charge of total {len(pairList)} year-month pairs.')
        return pairList
    else:
        print('Invalid Session ID.')

In [ ]:
YMList = getYearMonthPair(1, col='pickup', freq='daily')

In [ ]:
for year, month in YMList:
  thisTaxiTrip = TaxiTrip(year=year, month=month, colab=True, checkPointFreq=5000, cupy=True, daily=True, leave=True)
  thisTaxiTrip.getMonthlyData()
  thisTaxiTrip.exportDF()

### Local에서 CuPy를 이용하여 작업 시 예시

In [84]:
thisTaxiTrip = TaxiTrip(year=2009, month=1, colab=False, checkPointFreq=5000, cupy=True, daily=True, leave=True)

Found the check point. Last ID: 200
Loading the taxi trip data set...
Initializing the dataframe...
Initializing the arrays...
Done.


In [85]:
thisTaxiTrip.getMonthlyData(lastColID=250)

Year: 2009, Month: 1:   1%|          | 250/38523 [00:14<36:19, 17.56it/s]  

Reached the last ID.


In [87]:
thisTaxiTrip.exportDF()